# 🕹️ Retro Arcade — Check & Score

> Build stamp: **2026-05-29 16:08:35**

Reads the **PBIR definition** of your report via `sempy.fabric.get_report_definition`,
grades each of the 5 levels, assigns a rank, and mints your signed badge.

Re-run any time — the report is fetched fresh every run.


## Step 0 — Configure


In [ ]:
# ============================================================
# 👇 Edit these two values before running
# ============================================================
PLAYER_NAME  = "Your Name Here"           # name shown on the badge
REPORT_NAME  = "Arcade_Hall_Report"       # name you gave your report
# WORKSPACE is auto-detected (current workspace)
# ============================================================


## Step 1 — Install / import dependencies


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "--disable-pip-version-check", "PyJWT>=2.6.0"],
               check=False, capture_output=True)
try:
    import sempy.fabric as fabric
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "semantic-link"],
                   check=True)
    import sempy.fabric as fabric
import json, base64
from collections import Counter


## Step 2 — Fetch the report definition (PBIR)


In [ ]:
# get_report_definition returns the PBIR parts as a DataFrame with columns
# ['path', 'payload', 'payloadType'] (payload base64-encoded).
print(f"📥 Fetching '{REPORT_NAME}'...")
df_def = fabric.get_report_definition(report=REPORT_NAME)

# Normalize into dict {path: text}
parts = {}
for _, row in df_def.iterrows():
    p = row["path"]
    payload = row["payload"]
    ptype   = (row.get("payloadType") or "").lower() if hasattr(row, "get") else ""
    try:
        if ptype == "inlinebase64" or (payload and len(payload) % 4 == 0):
            raw = base64.b64decode(payload).decode("utf-8", errors="replace")
        else:
            raw = str(payload)
    except Exception:
        raw = str(payload)
    parts[p] = raw

print(f"✅ Got {len(parts)} PBIR parts.")
print("Sample paths:")
for p in list(parts.keys())[:10]:
    print(" ", p)


## Step 3 — Parse pages, visuals, theme, mobile layout


In [ ]:
def _json(path):
    if path in parts:
        try: return json.loads(parts[path])
        except Exception: return None
    return None

# ---- Pages list ----
pages_index = _json("definition/pages/pages.json") or {}
page_names  = []
if isinstance(pages_index.get("pageOrder"), list):
    page_names = list(pages_index["pageOrder"])
else:
    # fallback: scan folders
    for p in parts:
        if p.startswith("definition/pages/") and p.endswith("/page.json"):
            page_names.append(p.split("/")[2])
    page_names = sorted(set(page_names))

print(f"📄 Pages found: {len(page_names)}")
for pn in page_names: print("  ", pn)

pages_data = []
for pn in page_names:
    pjson = _json(f"definition/pages/{pn}/page.json") or {}
    # gather visuals
    visuals = []
    for p in parts:
        prefix = f"definition/pages/{pn}/visuals/"
        if p.startswith(prefix) and p.endswith("/visual.json"):
            vj = _json(p) or {}
            visuals.append(vj)
    pages_data.append({"name": pn, "json": pjson, "visuals": visuals})

# ---- Report-level (theme, mobile) ----
report_json = _json("definition/report.json") or {}
theme_paths = [p for p in parts if p.startswith("StaticResources/RegisteredResources/")
                                   and p.endswith(".json")]
has_custom_theme = bool(theme_paths)


## Step 4 — Grade the 5 levels


In [ ]:
SLICER_KEYS = ("slicer", "advancedSlicerVisual")

def visual_type(v):
    # PBIR shape varies; check common fields
    return (v.get("visual", {}).get("visualType")
            or v.get("visualContainerObjects", {}).get("visualType")
            or v.get("singleVisual", {}).get("visualType")
            or v.get("visualType")
            or "")

def page_display_title(pj):
    # PBIR shape: pj['displayName'] is the page title shown in the tab
    return pj.get("displayName") or ""

def page_has_background(pj):
    # check pj['objects']['background'] presence (color or image)
    objs = pj.get("objects") or {}
    bg   = objs.get("background")
    return bool(bg)

def page_kind(pj):
    # Tooltip / Drillthrough markers
    # pageBindings shape varies, but tooltips usually have 'pageBindings':{'visualName':...} or 'type':'tooltip'
    opt = pj.get("type") or pj.get("pageType") or ""
    if str(opt).lower() == "tooltip": return "tooltip"
    if str(opt).lower() == "drillthrough" or pj.get("filterConfig", {}).get("filters"):
        # presence of drillthrough filter
        for f in (pj.get("filterConfig", {}).get("filters") or []):
            if str(f.get("type", "")).lower() == "drillthrough":
                return "drillthrough"
    # mobile layout?
    return "regular"

def page_has_mobile(pj):
    # PBIR mobile layout is a separate JSON: definition/pages/<page>/mobile.json
    return any(p.startswith(f"definition/pages/{pj.get('name','')}/mobile") for p in parts)

# ---------------- Level 1 ---------------- #
n_pages = len(pages_data)
titled  = sum(1 for pd in pages_data if page_display_title(pd["json"]).strip()
              and not page_display_title(pd["json"]).strip().lower().startswith("page "))
bg      = sum(1 for pd in pages_data if page_has_background(pd["json"]))
L1 = 0
L1 += 10 if n_pages >= 3 else (5 if n_pages == 2 else 0)
L1 += 5  if titled >= max(3, n_pages) else (3 if titled >= 2 else 0)
L1 += 5  if bg >= max(3, n_pages) else (3 if bg >= 1 else 0)
L1 = min(20, L1)
print(f"🟢 L1 Foundation:      pages={n_pages}  titled={titled}  bg={bg}  →  {L1}/20")

# ---------------- Level 2 ---------------- #
all_visuals = [v for pd in pages_data for v in pd["visuals"]]
types = Counter(visual_type(v) for v in all_visuals if visual_type(v))
n_unique = sum(1 for t,c in types.items() if c >= 1)
has_card   = any(t in ("card","cardVisual","multiRowCard") for t in types)
has_slicer = any(t in SLICER_KEYS for t in types)
L2 = min(20, n_unique * 4)
if not has_card:   L2 = min(L2, 16)
if not has_slicer: L2 = min(L2, 16)
print(f"🟡 L2 Visuals:         types_seen={dict(types)}  unique={n_unique}  card={has_card}  slicer={has_slicer}  →  {L2}/20")

# ---------------- Level 3 ---------------- #
n_slicers = sum(c for t,c in types.items() if t in SLICER_KEYS)
# sync slicer: a slicer with syncSlicers configured (very approximate)
sync_count = 0
for v in all_visuals:
    if visual_type(v) in SLICER_KEYS:
        s = json.dumps(v)
        if '"syncGroup"' in s or '"syncSlicers"' in s:
            sync_count += 1
# edit-interactions: presence of "interactions" overrides at page level
interactions = sum(1 for pd in pages_data
                   if '"filters"' in json.dumps(pd["json"]) and '"hidden"' in json.dumps(pd["json"]))
L3 = 0
L3 += 7 if n_slicers >= 2 else (4 if n_slicers == 1 else 0)
L3 += 7 if sync_count >= 1 else 0
L3 += 6 if interactions >= 1 else 0
L3 = min(20, L3)
print(f"🟠 L3 Interactivity:   slicers={n_slicers}  sync={sync_count}  interactions={interactions}  →  {L3}/20")

# ---------------- Level 4 ---------------- #
bookmarks_json = _json("definition/bookmarks/bookmarks.json")
n_bookmarks = 0
if isinstance(bookmarks_json, dict):
    n_bookmarks = len(bookmarks_json.get("items", []))
else:
    # scan folder
    n_bookmarks = sum(1 for p in parts if p.startswith("definition/bookmarks/") and p.endswith("/bookmark.json"))
n_tooltips      = sum(1 for pd in pages_data if page_kind(pd["json"]) == "tooltip")
n_drillthrough  = sum(1 for pd in pages_data if page_kind(pd["json"]) == "drillthrough")
L4 = 0
L4 += 7 if n_bookmarks >= 1 else 0
L4 += 7 if n_tooltips >= 1 else 0
L4 += 6 if n_drillthrough >= 1 else 0
L4 = min(20, L4)
print(f"🟣 L4 Storytelling:    bookmarks={n_bookmarks}  tooltips={n_tooltips}  drillthrough={n_drillthrough}  →  {L4}/20")

# ---------------- Level 5 ---------------- #
# conditional formatting: look for "objects" with "dataBars" / "background" / "fontColor" with "fillRule"/"gradient"
cond_fmt = 0
for v in all_visuals:
    s = json.dumps(v)
    if any(k in s for k in ('"dataBars"', '"colorScale"', '"fillRule"', '"gradient"')):
        cond_fmt += 1
n_mobile = sum(1 for pd in pages_data if page_has_mobile(pd["json"]))
L5 = 0
L5 += 7 if has_custom_theme else 0
L5 += 7 if cond_fmt >= 1 else 0
L5 += 6 if n_mobile >= 1 else 0
L5 = min(20, L5)
print(f"🔵 L5 Polish:          theme={has_custom_theme}  condFmt={cond_fmt}  mobile={n_mobile}  →  {L5}/20")

TOTAL = L1 + L2 + L3 + L4 + L5
print()
print("=" * 60)
print(f"  TOTAL SCORE: {TOTAL}/100")
print("=" * 60)

if   TOTAL >= 100: RANK = "Kill Screen Survivor"
elif TOTAL >=  80: RANK = "Arcade Legend"
elif TOTAL >=  60: RANK = "High Roller"
elif TOTAL >=  40: RANK = "Quarter Muncher"
elif TOTAL >=  20: RANK = "Newbie"
else:              RANK = "Spectator"

FINAL_SCORE = TOTAL
FINAL_RANK  = RANK
print(f"  RANK: {RANK}")


## Step 5 — 🏅 Mint your shareable badge


In [ ]:
# ============================================================
# Retro Arcade — Badge issuance
# HMAC-signed URL for the GitHub Pages badge viewer
# ============================================================
import json, time, hmac, hashlib, base64
from IPython.display import display, Markdown, HTML

_BADGE_SECRET = b"fabric-arcade-badge-v1-7K9mP3xQ"
_BASE_URL     = "https://maenglar78.github.io/fabric-arcade"
_GAME_ID      = "retro-arcade"

def _b64u(b: bytes) -> str:
    return base64.urlsafe_b64encode(b).rstrip(b"=").decode("ascii")

def _issue(game_id, player, rank, score):
    payload = {"v": 1, "g": game_id, "p": str(player),
               "r": str(rank), "s": int(score), "t": int(time.time())}
    body = json.dumps(payload, separators=(",", ":"), sort_keys=True).encode()
    sig  = hmac.new(_BADGE_SECRET, body, hashlib.sha256).digest()
    return f"{_BASE_URL}/badge.html?t={_b64u(body)}.{_b64u(sig)}"

score = globals().get("FINAL_SCORE", 0)
rank  = globals().get("FINAL_RANK", "Spectator")

if score < 20:
    display(Markdown(
        f"### 🚧 Not yet eligible (score {score}/100)\n\n"
        f"Reach **at least 20 points** to earn the Newbie badge. "
        f"Re-open `02_Quest` for the level checklist."
    ))
elif PLAYER_NAME.strip() in ("", "Your Name Here"):
    display(Markdown(
        "### ✍️ Set your name first\n\n"
        "Edit `PLAYER_NAME` in **Step 0** and re-run the notebook."
    ))
else:
    url = _issue(_GAME_ID, PLAYER_NAME, rank, score)
    display(Markdown(
        f"### 🏅 Badge minted\n\n"
        f"**{PLAYER_NAME}** — *{rank}* · score **{score}/100**\n\n"
        f"🔗 **[Open your badge]({url})**\n\n"
        f"Click *Download PNG* / *Share on LinkedIn* on the badge page."
    ))
    display(HTML(f'<a href="{url}" target="_blank" '
                 f'style="display:inline-block;padding:10px 20px;border-radius:8px;'
                 f'background:linear-gradient(135deg,#ff006e,#8338ec);color:white;'
                 f'text-decoration:none;font-weight:600">🏅 Open my badge page</a>'))
